# Capability Lab — IP Seal Only v1
Use somente após `Capability_Lab_IP_Exact_Head_Gate_v3` retornar `5_OF_5=GREEN`. Este launcher não reroda os gates; apenas sela o head validado.


In [ ]:
from google.colab import userdata
from getpass import getpass
import os, pathlib, subprocess, hashlib, json
REPO='lucasmateus334-oss/lucas-capability-os'
BRANCH='governance/ip-independence-chain-of-title-v1'
VALIDATED_HEAD='020e5385c222f1ef727bd9adee0ee2d5e48cfb47'
WORK=pathlib.Path('/content/caplab-ip-seal')
def token():
    for k in ('GITHUB_TOKEN','CAPLAB_GITHUB_TOKEN','GH_TOKEN'):
        try:
            v=userdata.get(k)
            if v:return v
        except Exception:pass
    return getpass('GitHub token (não será exibido): ').strip()
t=token()
if not t: raise SystemExit('TOKEN_MISSING')
ask=pathlib.Path('/content/caplab_seal_askpass.sh')
ask.write_text('#!/bin/sh\ncase \"$1\" in *Username*) echo \"x-access-token\";; *) printf \"%s\\n\" \"$CAPLAB_GH_TOKEN\";; esac\n',encoding='utf-8')
ask.chmod(0o700)
env=os.environ.copy(); env['GIT_ASKPASS']=str(ask); env['GIT_TERMINAL_PROMPT']='0'; env['CAPLAB_GH_TOKEN']=t
try:
    if WORK.exists(): subprocess.run(['rm','-rf',str(WORK)],check=True)
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',f'https://github.com/{REPO}.git',str(WORK)],check=True,env=env)
    head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=WORK,text=True).strip()
    if head!=VALIDATED_HEAD: raise SystemExit(f'HEAD_DRIFT: validated={VALIDATED_HEAD} current={head}')
    excluded_exact={'SOURCE_MANIFEST.json','jobs/active_job.json'}; excluded_prefixes=('artifacts/','receipts/')
    tracked=subprocess.check_output(['git','ls-files'],cwd=WORK,text=True).splitlines()
    files=[]
    for rel in sorted(p for p in tracked if p not in excluded_exact and not p.startswith(excluded_prefixes)):
        p=WORK/rel
        if p.is_file() and not p.is_symlink():
            b=p.read_bytes(); files.append({'path':rel,'size':len(b),'sha256':hashlib.sha256(b).hexdigest()})
    manifest={'product':'Lucas Capability OS','checkpoint':f'operational-reconciliation-{head[:12]}','schemaVersion':2,'file_count':len(files),'files':files,'manifest_policy':{'excluded_exact':sorted(excluded_exact),'excluded_prefixes':list(excluded_prefixes),'reason':'runtime evidence and mutable job state are not canonical source'}}
    (WORK/'SOURCE_MANIFEST.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
    subprocess.run(['python','scripts/verify_source_manifest.py'],cwd=WORK,check=True)
    r=WORK/'receipts'; r.mkdir(exist_ok=True)
    receipt=r/'IP_EXACT_HEAD_COLAB_RECEIPT.md'
    receipt.write_text(f'''# Capability Lab — IP Exact-Head Colab Receipt\n\n- Validated source head: `{head}`\n- Validation method: Colab v3 isolated exact-head gate\n- Capability OS quality gate: GREEN\n- RC1 remaining regression tests: GREEN\n- Gemini Zero-Cost Runtime CI: GREEN\n- AI Studio Zero-Cost VPE: GREEN\n- Capability UI gate: GREEN\n- Result: **5/5 GREEN**\n- Seal-only step: manifest + receipt persisted after operator-confirmed GREEN, without re-executing gates.\n\nNota de fluxo — Este receipt registra o exact-head validado e a persistência final da evidência, mantendo `SOURCE_MANIFEST.json` e `receipts/` fora do conjunto canônico de source.\n''',encoding='utf-8')
    subprocess.run(['git','config','user.name','Lucas Mateus'],cwd=WORK,check=True)
    subprocess.run(['git','config','user.email','lucasmateus334@gmail.com'],cwd=WORK,check=True)
    subprocess.run(['git','add','SOURCE_MANIFEST.json'],cwd=WORK,check=True)
    subprocess.run(['git','add','-f','receipts/IP_EXACT_HEAD_COLAB_RECEIPT.md'],cwd=WORK,check=True)
    subprocess.run(['git','commit','-m','chore: persist validated IP exact-head seal','-m','O que isso faz: persiste o manifesto canônico e o receipt do head 5/5 GREEN validado no Colab.'],cwd=WORK,check=True)
    subprocess.run(['git','push','origin',f'HEAD:{BRANCH}'],cwd=WORK,check=True,env=env)
    new_head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=WORK,text=True).strip()
    print(f'CAPLAB_SEAL=GREEN\nVALIDATED_HEAD={head}\nSEALED_COMMIT={new_head}')
finally:
    env.pop('CAPLAB_GH_TOKEN',None); ask.unlink(missing_ok=True); t=None
# O que isso faz: sela apenas a evidência do exact-head já validado, sem repetir os cinco gates.
